In [ ]:
# Notebook 11: Evaluation Suite

!pip install bert-score rouge-score nltk -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
from tqdm import tqdm
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import nltk

nltk.download('punkt', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


Using device: cpu


In [ ]:

# Load models for comparison
BASE_MODEL_DIR = "gpt2-medium"
SFT_MODEL_DIR = BASE_DIR / "models/sft_model/final"

print("Loading models...")

# Base model
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_DIR).to(DEVICE)

# SFT model
sft_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR)
sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_DIR).to(DEVICE)

print("All models loaded")


Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

All models loaded


In [ ]:

# Test queries
test_queries = [
    "What are transformers in deep learning?",
    "Explain RLHF and how it works",
    "What is the attention mechanism?",
    "How do diffusion models generate images?",
    "What are the benefits of LoRA fine-tuning?",
    "Explain constitutional AI",
    "What is retrieval augmented generation?",
    "How does BERT work?",
    "What are the differences between GPT and BERT?",
    "Explain the concept of embeddings in NLP"
]

# Reference answers (simplified for evaluation)
reference_answers = [
    "Transformers are neural network architectures that use self-attention mechanisms to process sequences. They were introduced in the paper 'Attention is All You Need' and have become the foundation for models like BERT and GPT.",
    "RLHF (Reinforcement Learning from Human Feedback) is a technique to align AI models with human preferences. It involves training a reward model on human preference data, then using reinforcement learning to optimize the model.",
    "Attention mechanism allows models to focus on relevant parts of the input when processing sequences. It computes weights that determine how much each input element should contribute to the output.",
    "Diffusion models generate images by learning to reverse a gradual noising process. They start with random noise and iteratively denoise it to create coherent images.",
    "LoRA (Low-Rank Adaptation) allows efficient fine-tuning by adding small trainable matrices to frozen pretrained weights. This dramatically reduces the number of trainable parameters.",
    "Constitutional AI uses a set of principles to critique and revise model outputs. It helps ensure AI systems are helpful, harmless, and honest through self-improvement.",
    "RAG combines retrieval systems with language models. It retrieves relevant documents from a knowledge base and uses them as context for generation.",
    "BERT is a bidirectional transformer model pretrained on masked language modeling. It learns contextual representations by predicting masked tokens.",
    "GPT is autoregressive and generates text left-to-right, while BERT is bidirectional and better for understanding tasks. GPT is for generation, BERT for comprehension.",
    "Embeddings are dense vector representations of words or tokens that capture semantic meaning. Similar words have similar embedding vectors."
]

In [ ]:

# Generation function
def generate_response(query, model, tokenizer, max_new_tokens=100):
    """Generate response from a model"""
    inputs = tokenizer(query, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace(query, "").strip()
    return response


In [ ]:


# Generate responses from all models
print("\nGenerating responses...")

results = {
    'base': [],
    'sft': []
}

for query in tqdm(test_queries, desc="Generating"):
    results['base'].append(generate_response(query, base_model, base_tokenizer))
    results['sft'].append(generate_response(query, sft_model, sft_tokenizer))

print("Generation complete")


Generating responses...


Generating: 100%|██████████| 10/10 [06:00<00:00, 36.08s/it]

Generation complete


In [ ]:


# Evaluation metrics
def calculate_rouge(predictions, references):
    """Calculate ROUGE scores"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    return {
        'rouge1': np.mean(rouge1_scores),
        'rouge2': np.mean(rouge2_scores),
        'rougeL': np.mean(rougeL_scores)
    }

def calculate_bertscore(predictions, references):
    """Calculate BERTScore"""
    P, R, F1 = bert_score(predictions, references, lang='en', verbose=False)
    return {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item()
    }

def calculate_length_stats(texts):
    """Calculate text length statistics"""
    lengths = [len(t.split()) for t in texts]
    return {
        'mean_length': np.mean(lengths),
        'median_length': np.median(lengths),
        'min_length': np.min(lengths),
        'max_length': np.max(lengths)
    }


In [ ]:


# Evaluate all models
print("\nEvaluating models...")

evaluation_results = {}

for model_name in ['base', 'sft']:
    print(f"\nEvaluating {model_name.upper()} model...")

    predictions = results[model_name]

    # ROUGE scores
    rouge_scores = calculate_rouge(predictions, reference_answers)

    # BERTScore
    bert_scores = calculate_bertscore(predictions, reference_answers)

    # Length stats
    length_stats = calculate_length_stats(predictions)

    evaluation_results[model_name] = {
        'rouge': rouge_scores,
        'bertscore': bert_scores,
        'length': length_stats
    }

    print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
    print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")
    print(f"BERTScore F1: {bert_scores['f1']:.4f}")
    print(f"Avg length: {length_stats['mean_length']:.1f} words")


Evaluating models...

Evaluating BASE model...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE-1: 0.1395
ROUGE-L: 0.1027
BERTScore F1: 0.8286
Avg length: 77.1 words

Evaluating SFT model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE-1: 0.1521
ROUGE-L: 0.1131
BERTScore F1: 0.8215
Avg length: 72.6 words


In [ ]:



# Comparison table
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

print(f"\n{'Metric':<20} {'Base':<15} {'SFT':<15}")
print("-" * 80)

# ROUGE-1
print(f"{'ROUGE-1':<20} {evaluation_results['base']['rouge']['rouge1']:<15.4f} {evaluation_results['sft']['rouge']['rouge1']:<15.4f}")

# ROUGE-L
print(f"{'ROUGE-L':<20} {evaluation_results['base']['rouge']['rougeL']:<15.4f} {evaluation_results['sft']['rouge']['rougeL']:<15.4f}")

# BERTScore
print(f"{'BERTScore F1':<20} {evaluation_results['base']['bertscore']['f1']:<15.4f} {evaluation_results['sft']['bertscore']['f1']:<15.4f}")

# Length
print(f"{'Avg Length (words)':<20} {evaluation_results['base']['length']['mean_length']:<15.1f} {evaluation_results['sft']['length']['mean_length']:<15.1f}")



MODEL COMPARISON

Metric               Base            SFT            
--------------------------------------------------------------------------------
ROUGE-1              0.1395          0.1521         
ROUGE-L              0.1027          0.1131         
BERTScore F1         0.8286          0.8215         
Avg Length (words)   77.1            72.6           


In [ ]:

# Save results
OUTPUT_DIR = BASE_DIR / "results/evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Convert numpy types to Python types
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

evaluation_results_serializable = convert_to_serializable(evaluation_results)

# Save detailed results
with open(OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(evaluation_results_serializable, f, indent=2)

# Save generated responses
with open(OUTPUT_DIR / 'generated_responses.json', 'w') as f:
    json.dump({
        'queries': test_queries,
        'references': reference_answers,
        'responses': results
    }, f, indent=2)

print(f"\nResults saved to {OUTPUT_DIR}")


Results saved to /content/drive/MyDrive/multi_agent_rag_project/results/evaluation


In [ ]:

# Qualitative examples
print("\n" + "="*80)
print("QUALITATIVE EXAMPLES")
print("="*80)

for i in range(min(3, len(test_queries))):
    print(f"\n{'='*80}")
    print(f"Query: {test_queries[i]}")
    print(f"{'='*80}")

    print(f"\nBase Model:")
    print(f"{results['base'][i]}")

    print(f"\nSFT Model:")
    print(f"{results['sft'][i]}")

    print(f"\nReference:")
    print(f"{reference_answers[i]}")




QUALITATIVE EXAMPLES

Query: What are transformers in deep learning?

Base Model:
Transformer is a technology which is used to process image data from a large number of different sources. The key concept is to combine the input from these sources into a single image with the desired characteristics. That is why you see many applications of transformers across different industries.

Why is deep learning so critical today?



The fundamental goal of deep learning is to learn by doing and the application of deep neural networks has seen its growth in order to achieve this goal. Deep learning

SFT Model:
###2. Transformers are a component of neural networks. They are used to generate a new representation from each input. In the following model, a transform is a transformation from the input image to the new image. To learn the model we first take the training set. Then, for each training epoch, we compute the posterior distribution and test the model using the standard error. In this exam